In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import os
import glob
for dirname, _, filenames in os.walk('/spam-detection'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [2]:
excel_files = glob.glob("*.xlsx")
excel_files

['2024- text comments.xlsx.xlsx',
 'Cycle 1- text comments.xlsx.xlsx',
 'JAN 2025- text comments.xlsx.xlsx',
 'MAR 2025 Reporting- text comments.xlsx.xlsx']

In [3]:
data = pd.concat([pd.read_excel(file) for file in excel_files]).reset_index()

In [4]:
# Fill NaN values in 'comments' with empty string
data.fillna({"comments": ""}, inplace=True)

In [5]:
# Define our features (X) and the target (y)
X = data[['comments', 'sentiment category']]
y = data['hide comment']

In [6]:
# This is crucial to evaluate how well our model generalizes to new, unseen data.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [7]:
# This technique reflects the importance of a word in a document within a collection of documents.
text_processor = TfidfVectorizer()

In [8]:
categorical_processor = OneHotEncoder(handle_unknown='ignore')

In [9]:

# Combine the processors into a single ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('text', text_processor, 'comments'),
        ('category', categorical_processor, ['sentiment category'])
    ],
    remainder='passthrough' # Keep other columns if any (none in this case)
)


In [10]:
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

In [11]:
model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('text', TfidfVectorizer(),
                                                  'comments'),
                                                 ('category',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['sentiment category'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced',
                                        random_state=42))])

In [12]:
y_pred = model_pipeline.predict(X_test)

In [13]:
print(confusion_matrix(y_test, y_pred))

[[424  14]
 [  2 166]]


In [14]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       1.00      0.97      0.98       438
        True       0.92      0.99      0.95       168

    accuracy                           0.97       606
   macro avg       0.96      0.98      0.97       606
weighted avg       0.98      0.97      0.97       606



In [15]:
confusion_matrix(y_test, y_pred)

array([[424,  14],
       [  2, 166]])

In [16]:
print(accuracy_score(y_test, y_pred))

0.9735973597359736


In [17]:
# Create a DataFrame to compare actual and predicted values
comparison_df = pd.DataFrame({
    'comments': X_test['comments'],
    'sentiment_category': X_test['sentiment category'],
    'actual': y_test,
    'predicted': y_pred
})

# Identify false positives
false_positives = comparison_df[(comparison_df['actual'] == 0) & (comparison_df['predicted'] == 1)]

# Print false positives
print("False Positives:")
false_positives

False Positives:


,comments,sentiment_category,actual,predicted
1869,Great changes,neutral,False,True
993,无,neutral,False,True
755,无,neutral,False,True
931,无,neutral,False,True
353,engage teams earlier,neutral,False,True
654,无,neutral,False,True
1005,ok,NaN,False,True
707,无,neutral,False,True
1829,nothing changed,neutral,False,True
926,无,neutral,False,True


In [18]:
false_negativess = comparison_df[(comparison_df['actual'] == 1) & (comparison_df['predicted'] == 0)]
false_negativess

,comments,sentiment_category,actual,predicted
2243,"The questions here are too broad, I find that ...",NaN,True,False
1675,no comments but as outside DD&T some of the te...,NaN,True,False


In [15]:
import joblib

joblib.dump(model_pipeline, "comment_hide_classifier.joblib")


['comment_hide_classifier.joblib']

import joblib

joblib.dump(model, 'model.pkl')
joblib.dump(X.columns, 'model_features.pkl')

model = joblib.load('model.pkl')
model_features = joblib.load('model_features.pkl')

# Function to make predictions
def predict_hide_comment(comments, sentiment_category) -> bool:
    # Create a DataFrame for the input data
    input_data = pd.DataFrame({
        # 'question code': [question_code],
        'comments': [comments],
        'sentiment category': [sentiment_category],
    })
    
    # Convert categorical data to numerical data for 'question code'
    input_data = pd.get_dummies(input_data, columns=['sentiment category'])
    
    # Vectorize 'comments' column using the loaded TF-IDF vectorizer

    comments_tfidf = vectorizer.transform(input_data['comments']).toarray()
    
    # Drop the original 'comments' column and add the TF-IDF features
    input_data = input_data.drop(columns=['comments'])
    input_data = pd.concat([input_data, pd.DataFrame(comments_tfidf, index=input_data.index)], axis=1)

    # Reindex input_data to match the columns used during training
    # Fill any missing columns with 0
    input_data = input_data.reindex(columns=model_features, fill_value=0)
    
    input_data.columns = input_data.columns.astype(str)
    
    # Make prediction
    prediction = model.predict(input_data)
    
    return prediction[0]


# Example usage
# question_code = 'dq|change_observed'
comments = "There has been an introduction to AI solutions and tools that have helped with workflow."
sentiment_category = "neutral"

result = predict_hide_comment(comments, sentiment_category)
print(f"Prediction: {result}")